# 🧃 OWASP Juice Shop — Security Testing Lab
### v20.0.0 · Hosted in Google Colab · Exposed via Cloudflare Tunnel

---

**Author:** Chris Gillham  
**Purpose:** Spin up a live OWASP Juice Shop instance for hands-on web application security testing  
**Framework References:** OWASP Top 10 (2021), OWASP WSTG, OWASP Juice Shop v20.0.0

---

## ⚠️ Legal & Ethical Notice

> This lab is for **authorized security education and testing only**.  
> OWASP Juice Shop is an intentionally vulnerable application — **never expose credentials or real data** in this environment.  
> The public tunnel URL is ephemeral and session-scoped. Treat it as a private lab environment.

---

## 🏗️ Architecture

```
┌─────────────────────────────────────────────────────────────┐
│  Google Colab VM                                            │
│                                                             │
│  ┌──────────────────────┐    ┌──────────────────────────┐  │
│  │  Juice Shop (Node 22) │───▶│  cloudflared tunnel      │  │
│  │  localhost:3000       │    │  → random.trycloudflare  │  │
│  └──────────────────────┘    └──────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘
                                          │
                                    Public HTTPS URL
                                    (printed in Cell 4)
```

## 📋 Cells
| Cell | Description |
|------|-------------|
| **Cell 1** | Install Node.js 22 LTS (v20 dropped support for Node 20) |
| **Cell 2** | Download Juice Shop v20.0.0 pre-built tarball from GitHub |
| **Cell 3** | Install cloudflared (no account or auth token needed) |
| **Cell 4** | 🚀 Launch — starts Juice Shop + tunnel, prints public URL |
| **Cell 5** | Status check & log viewer |
| **Cell 6** | Challenge quick-reference (OWASP Top 10 mapped, v20 AI challenges) |
| **Cell 7** | 🛑 Shutdown |

---
**Run all cells in order. The live URL appears at the end of Cell 4.**

> **Why tarball instead of npm?**  
> The `@juice-shop/juice-shop` scoped package is **not published to the public npm registry**.  
> The official install method is the pre-built release tarball from GitHub, which includes all native binary dependencies pre-compiled for the target Node version.

## ⚙️ Cell 1 — Install Node.js 22 LTS

Juice Shop v20.0.0 requires **Node.js 22+** — Node 20 support was dropped in this release.

In [15]:
# ============================================================
# Cell 1: Install Node.js 22 LTS
# Juice Shop v20.0.0 requires Node 22+.
# Colab ships with Node 18/20 depending on the runtime image,
# so we always install Node 22 via NodeSource.
# ============================================================
import subprocess, shutil, os, builtins

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("=" * 60)
print("  OWASP Juice Shop v20 Lab — Node.js 22 Setup")
print("=" * 60)

node_ver = run("node --version 2>/dev/null")
major    = int(node_ver.lstrip('v').split('.')[0]) if node_ver else 0
print(f"  Current Node.js : {node_ver or 'not found'} (major={major})")

if major >= 22:
    print(f"  ✅  Node {node_ver} already satisfies the v22+ requirement.")
else:
    print(f"  ⬆️   Installing Node.js 22 LTS via NodeSource...")
    subprocess.run(
        "curl -fsSL https://deb.nodesource.com/setup_22.x | bash - 2>&1",
        shell=True
    )
    subprocess.run("apt-get install -y nodejs 2>&1", shell=True)
    node_ver = run("node --version")
    major    = int(node_ver.lstrip('v').split('.')[0]) if node_ver else 0
    if major >= 22:
        print(f"  ✅  Node.js upgraded to {node_ver}")
    else:
        raise RuntimeError(f"Node 22 install failed — still on {node_ver}")

npm_ver = run("npm --version")
print(f"  ✅  npm         : v{npm_ver}")

# Disk & network
disk    = shutil.disk_usage("/")
free_gb = disk.free / (1024**3)
print(f"  {'✅' if free_gb > 1.5 else '⚠️ '}  Disk free  : {free_gb:.1f} GB")

gh_ok = run("curl -s --max-time 8 -o /dev/null -w '%{http_code}' https://github.com")
cf_ok = run("curl -s --max-time 8 -o /dev/null -w '%{http_code}' https://api.cloudflare.com")
print(f"  {'✅' if gh_ok == '200' else '❌'}  GitHub     : HTTP {gh_ok}")
print(f"  {'✅' if cf_ok == '200' else '❌'}  Cloudflare : HTTP {cf_ok}")

WORK_DIR = "/opt/juiceshop"
os.makedirs(WORK_DIR, exist_ok=True)
print(f"  ✅  Work dir   : {WORK_DIR}")

# Persist for subsequent cells
builtins.WORK_DIR    = WORK_DIR
builtins.JUICE_PORT  = 3000
builtins.NODE_VER    = node_ver
builtins.NODE_MAJOR  = major

print()
print("  ✅  Cell 1 complete — proceed to Cell 2.")
print("=" * 60)

  OWASP Juice Shop v20 Lab — Node.js 22 Setup
  Current Node.js : v22.22.2 (major=22)
  ✅  Node v22.22.2 already satisfies the v22+ requirement.
  ✅  npm         : v10.9.7
  ✅  Disk free  : 86.2 GB
  ✅  GitHub     : HTTP 200
  ❌  Cloudflare : HTTP 301
  ✅  Work dir   : /opt/juiceshop

  ✅  Cell 1 complete — proceed to Cell 2.


## 📦 Cell 2 — Download Juice Shop v20.0.0

Downloads the pre-built Linux x64 tarball from the official GitHub release page.  
All native dependencies (sqlite3, libxmljs2) are pre-compiled for the target Node version — **no build step required**.

In [16]:
import subprocess, os, time, builtins

JUICE_VERSION = "20.0.0"
WORK_DIR      = builtins.WORK_DIR
BASE_URL      = f"https://github.com/juice-shop/juice-shop/releases/download/v{JUICE_VERSION}"

def make_asset(node_n):
    name = f"juice-shop-{JUICE_VERSION}_node{node_n}_linux_x64.tgz"
    return name, f"{BASE_URL}/{name}", f"/tmp/{name}"

def asset_exists(url):
    """Fast HEAD check — does this asset exist on GitHub?"""
    r = subprocess.run(
        f"curl -fsIL --max-time 15 -o /dev/null -w '%{{http_code}}' '{url}'",
        shell=True, capture_output=True, text=True
    )
    return r.stdout.strip() == "200"

print("=" * 60)
print(f"  Juice Shop v{JUICE_VERSION} — Pre-Built Tarball Install")
print("=" * 60)

JUICE_ROOT_DIR = os.path.join(WORK_DIR, f"juice-shop_{JUICE_VERSION}") # This is the extracted root
APP_JS_PATH    = os.path.join(JUICE_ROOT_DIR, "app.js")

ASSET_NAME   = None
DOWNLOAD_URL = None
TARBALL_PATH = None

if os.path.exists(APP_JS_PATH):
    print(f"  ✅  Already extracted at {JUICE_ROOT_DIR} — skipping.")
else:
    # ── Probe: try node24, node22, node20 in order ────────────
    # Prioritize the installed Node.js version
    PROBE_ORDER = [builtins.NODE_MAJOR] + [x for x in [24, 22, 20] if x != builtins.NODE_MAJOR]
    print(f"  🔍 Probing for available node variant ({PROBE_ORDER})...")

    for node_n in PROBE_ORDER:
        name, url, path = make_asset(node_n)
        # Use cached tarball if already downloaded
        if os.path.exists(path) and os.path.getsize(path) > 10_000_000:
            print(f"  ✅  Cached: {name}")
            ASSET_NAME, DOWNLOAD_URL, TARBALL_PATH = name, url, path
            break
        print(f"     Checking node{node_n}... ", end="", flush=True)
        if asset_exists(url):
            print("found!")
            ASSET_NAME, DOWNLOAD_URL, TARBALL_PATH = name, url, path
            break
        else:
            print("not found")

    if not ASSET_NAME:
        raise RuntimeError(
            f"No pre-built tarball found for node {PROBE_ORDER}.\n"
            f"Check: https://github.com/juice-shop/juice-shop/releases/tag/v{JUICE_VERSION}\n"
            f"and update PROBE_ORDER above with the correct node major."
        )

    print(f"  ✅  Asset       : {ASSET_NAME}")
    print(f"     Source      : {DOWNLOAD_URL}")
    print(f"     Extract to  : {WORK_DIR}")
    print()

    # ── Download ──────────────────────────────────────────────
    if not (os.path.exists(TARBALL_PATH) and os.path.getsize(TARBALL_PATH) > 10_000_000):
        print(f"  ⏳ Downloading {ASSET_NAME}...")
        t0 = time.time()
        rc = subprocess.run(
            f"curl -fL --progress-bar '{DOWNLOAD_URL}' -o '{TARBALL_PATH}'",
            shell=True
        ).returncode
        if rc != 0 or not os.path.exists(TARBALL_PATH):
            raise RuntimeError(f"Download failed (rc={rc})")
        elapsed = time.time() - t0
        size_mb = os.path.getsize(TARBALL_PATH) / (1024**2)
        print(f"  ✅  Downloaded {size_mb:.0f} MB in {elapsed:.0f}s")
    else:
        print(f"  ✅  Tarball already cached")

    # ── Extract ───────────────────────────────────────────────
    print(f"  ⏳ Extracting...")
    rc = subprocess.run(
        f"tar -xzf '{TARBALL_PATH}' -C '{WORK_DIR}'", shell=True
    ).returncode
    if rc != 0:
        raise RuntimeError("tar extraction failed")

    # Locate app.js within the extracted root, and update APP_JS_PATH.
    # JUICE_ROOT_DIR remains the top-level directory for npm start cwd.
    if not os.path.exists(APP_JS_PATH):
        found_app_js = subprocess.run(
            f"find '{JUICE_ROOT_DIR}' -maxdepth 3 -name 'app.js' 2>/dev/null | head -1",
            shell=True, capture_output=True, text=True
        ).stdout.strip()
        if found_app_js:
            APP_JS_PATH = found_app_js # Update APP_JS_PATH to the correct path
        else:
            raise FileNotFoundError(f"Cannot find app.js under {JUICE_ROOT_DIR}")
    print(f"  ✅  Extraction complete")

print(f"  ✅  Juice Shop dir : {JUICE_ROOT_DIR}") # This is passed to Cell 4 as cwd for npm start
print(f"  ✅  Entry point    : {APP_JS_PATH}")
print(f"  ✅  Launch method  : npm start  (per official docs)")
builtins.JUICE_APP_JS = APP_JS_PATH
builtins.JUICE_DIR    = JUICE_ROOT_DIR

# ── Native module sanity check ────────────────────────────────
# Check if sqlite3 module can be loaded by Node.js within the Juice Shop context
chk = subprocess.run(
    ["node", "-e", "require('sqlite3')"],
    cwd=JUICE_ROOT_DIR,
    capture_output=True,
    text=True
)

if chk.returncode == 0:
    print(f"  ✅  sqlite3 native module loads OK")
else:
    print(f"  ⚠️   sqlite3 load warning: {chk.stderr[:200].strip()}")
    print("      If you see a NODE_MODULE_VERSION mismatch, the tarball")
    print("      targets a different Node major than currently installed.")

print()
print("  ✅  Cell 2 complete — proceed to Cell 3.")
print("=" * 60)


  Juice Shop v20.0.0 — Pre-Built Tarball Install
  🔍 Probing for available node variant ([22, 24, 20])...
  ✅  Cached: juice-shop-20.0.0_node22_linux_x64.tgz
  ✅  Asset       : juice-shop-20.0.0_node22_linux_x64.tgz
     Source      : https://github.com/juice-shop/juice-shop/releases/download/v20.0.0/juice-shop-20.0.0_node22_linux_x64.tgz
     Extract to  : /opt/juiceshop

  ✅  Tarball already cached
  ⏳ Extracting...
  ✅  Extraction complete
  ✅  Juice Shop dir : /opt/juiceshop/juice-shop_20.0.0
  ✅  Entry point    : /opt/juiceshop/juice-shop_20.0.0/build/app.js
  ✅  Launch method  : npm start  (per official docs)
  ✅  sqlite3 native module loads OK

  ✅  Cell 2 complete — proceed to Cell 3.


## 🌐 Cell 3 — Install Cloudflare Tunnel (cloudflared)

Downloads the `cloudflared` binary. Uses `--quick-tunnel` — **no account or auth token required**.

In [17]:
# ============================================================
# Cell 3: Install cloudflared
# ============================================================
import subprocess, os, builtins

CF_BIN = "/usr/local/bin/cloudflared"
CF_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

print("=" * 60)
print("  Installing cloudflared...")
print("=" * 60)

if os.path.exists(CF_BIN):
    ver = subprocess.run([CF_BIN, "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  ✅  Already installed: {ver}")
else:
    print(f"  ⏳ Downloading cloudflared...")
    rc1 = subprocess.run(f"curl -fsSL '{CF_URL}' -o '{CF_BIN}'", shell=True).returncode
    rc2 = subprocess.run(f"chmod +x '{CF_BIN}'", shell=True).returncode
    if rc1 != 0 or rc2 != 0:
        raise RuntimeError("cloudflared install failed")
    ver = subprocess.run([CF_BIN, "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  ✅  Installed: {ver}")

builtins.CF_BIN = CF_BIN

print()
print("  ✅  Cell 3 complete — proceed to Cell 4.")
print("=" * 60)

  Installing cloudflared...
  ✅  Already installed: cloudflared version 2026.5.0 (built 2026-05-13-11:24 UTC)

  ✅  Cell 3 complete — proceed to Cell 4.


## 🚀 Cell 4 — Launch Juice Shop + Public Tunnel

Starts both processes and prints the public URL.

> **Cloudflare interstitial:** First visit shows a safety page — click **Visit Site** to proceed.

In [32]:
# ============================================================
# Cell 4: Launch Juice Shop + Cloudflare Tunnel
# Uses 'npm start' per official docs (not 'node app.js').
# ============================================================
import subprocess, time, re, os, sys, builtins, socket
from google.colab import userdata

JUICE_DIR = builtins.JUICE_DIR
PORT      = builtins.JUICE_PORT
CF_BIN    = builtins.CF_BIN
LOG_JS    = "/tmp/juiceshop.log"
LOG_CF    = "/tmp/cloudflared.log"

# Kill any previous instances
for pat in ["npm.*start", "node.*bin/juice-shop", "cloudflared", "node.*build/app.js"]: # Added direct node app.js to kill list
    subprocess.run(f"pkill -f '{pat}' 2>/dev/null", shell=True)
time.sleep(1)

print("=" * 60)
print("  Launching OWASP Juice Shop v20.0.0...")
print("=" * 60)
print(f"  Dir  : {JUICE_DIR}")
print(f"  Port : {PORT}")
print(f"  Cmd  : node {builtins.JUICE_APP_JS} (direct execution)") # Updated command display
print()

# ── Directly start Node.js application ──────────────────────
env = os.environ.copy()
env["PORT"]     = str(PORT)
env["NODE_ENV"] = "production"
env["ALCHEMY_API_KEY"] = 'ALCHEMY_API_KEY'

#Need to figure out how to inject an external LLM API URL and key.
#env["LLM_API_KEY"] = 'MISTRAL_API_KEY'
#env["llmApiUrl"] = 'https://api.mistral.ai/v1'

with open(LOG_JS, "w") as f:
    js_proc = subprocess.Popen(
        ["node", builtins.JUICE_APP_JS],
        cwd=JUICE_DIR,
        env=env,
        stdout=f,
        stderr=subprocess.STDOUT
    )

print(f"  ⏳ Juice Shop starting (PID {js_proc.pid})...")
print("     Startup optimised ~30% in v20 — expect 10–25s")

js_ready = False
for _ in range(90):
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=1):
            js_ready = True
            break
    except Exception:
        pass
    if js_proc.poll() is not None:
        print(f"\n  ❌  Node process exited early (rc={js_proc.returncode}).") # Updated message
        print("      Run Cell 5 to inspect logs.")
        break
    time.sleep(1)
    sys.stdout.write(".")
    sys.stdout.flush()

if not js_ready:
    js_proc.terminate()
    raise RuntimeError("Juice Shop startup timed out — run Cell 5 to inspect logs.")

print(f"\n  ✅  Juice Shop ready on http://localhost:{PORT}")

# ── Start cloudflared tunnel ─────────────────────────────────
print(f"\n  ⏳ Starting Cloudflare quick tunnel...")

with open(LOG_CF, "w") as f:
    cf_proc = subprocess.Popen(
        [CF_BIN, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
        stdout=f,
        stderr=subprocess.STDOUT
    )

public_url  = None
url_pattern = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')

for _ in range(45):
    try:
        with open(LOG_CF) as f:
            match = url_pattern.search(f.read())
        if match:
            public_url = match.group(0)
            break
    except Exception:
        pass
    time.sleep(1)
    sys.stdout.write(".")
    sys.stdout.flush()

builtins.JS_PROC = js_proc
builtins.CF_PROC = cf_proc

print()
print("=" * 60)

if public_url:
    print()
    print("  🎯  JUICE SHOP IS LIVE!")
    print()
    print(f"  🌐  Public URL   : {public_url}")
    print(f"  📊  Score board  : {public_url}/#/score-board")
    print()
    print("  📋  Default accounts:")
    print("       admin@juice-sh.op    : admin123")
    print("       jim@juice-sh.op      : ncc-1701")
    print("       bender@juice-sh.op   : OhWellHellW0rld")
    print("       basil@juice-sh.op    : (new in v20 — try common passwords)")
    print()
    print("  ⚠️   First visit: click 'Visit Site' past the Cloudflare interstitial.")
    print()
    print(f"  ╔══════════════════════════════════════════════════════╗")
    print(f"  ║  {public_url:<52}  ║")
    print(f"  ╚══════════════════════════════════════════════════════╝")
else:
    print("  ⚠️   Could not extract public URL — run Cell 5 to read cloudflared log.")
    print(f"      Local: http://localhost:{PORT}")

print()
print(f"  Juice Shop PID   : {js_proc.pid}")
print(f"  cloudflared PID  : {cf_proc.pid}")
print(f"  Logs: {LOG_JS}  /  {LOG_CF}")
print("=" * 60)


  Launching OWASP Juice Shop v20.0.0...
  Dir  : /opt/juiceshop/juice-shop_20.0.0
  Port : 3000
  Cmd  : node /opt/juiceshop/juice-shop_20.0.0/build/app.js (direct execution)

  ⏳ Juice Shop starting (PID 14326)...
     Startup optimised ~30% in v20 — expect 10–25s
.......
  ✅  Juice Shop ready on http://localhost:3000

  ⏳ Starting Cloudflare quick tunnel...
....

  🎯  JUICE SHOP IS LIVE!

  🌐  Public URL   : https://consistency-political-distribute-survival.trycloudflare.com
  📊  Score board  : https://consistency-political-distribute-survival.trycloudflare.com/#/score-board

  📋  Default accounts:
       admin@juice-sh.op    : admin123
       jim@juice-sh.op      : ncc-1701
       bender@juice-sh.op   : OhWellHellW0rld
       basil@juice-sh.op    : (new in v20 — try common passwords)

  ⚠️   First visit: click 'Visit Site' past the Cloudflare interstitial.

  ╔══════════════════════════════════════════════════════╗
  ║  https://consistency-political-distribute-survival.trycloudflare

## 📡 Cell 5 — Status Check & Log Viewer

Re-run any time to check process health or diagnose startup issues.

In [33]:
# ============================================================
# Cell 5: Status Check & Log Viewer
# ============================================================
import subprocess, re, os, socket, builtins

PORT   = getattr(builtins, 'JUICE_PORT', 3000)
LOG_JS = "/tmp/juiceshop.log"
LOG_CF = "/tmp/cloudflared.log"

print("=" * 60)
print("  Status Check")
print("=" * 60)

r    = subprocess.run("pgrep -f 'node.*app.js' | head -1", shell=True, capture_output=True, text=True)
r2   = subprocess.run("pgrep -f cloudflared | head -1",    shell=True, capture_output=True, text=True)
js_pid = r.stdout.strip()
cf_pid = r2.stdout.strip()

print(f"  Juice Shop : {'✅ PID ' + js_pid if js_pid else '❌ not running'}")
try:
    with socket.create_connection(("127.0.0.1", PORT), timeout=1):
        print(f"  Port {PORT}   : ✅ accepting connections")
except Exception:
    print(f"  Port {PORT}   : ❌ not listening")
print(f"  cloudflared: {'✅ PID ' + cf_pid if cf_pid else '❌ not running'}")

if os.path.exists(LOG_CF):
    with open(LOG_CF) as f:
        content = f.read()
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', content)
    if m:
        print(f"\n  🌐  Public URL  : {m.group(0)}")
        print(f"  📊  Score board : {m.group(0)}/#/score-board")
    else:
        print("  ⚠️   Public URL not yet in cloudflared log.")

print("\n  ── Juice Shop log (last 20 lines) ──────────────────")
if os.path.exists(LOG_JS):
    for line in subprocess.run(f"tail -20 {LOG_JS}", shell=True, capture_output=True, text=True).stdout.splitlines():
        print(f"  {line}")
else:
    print("  (not found — run Cell 4 first)")

print("\n  ── cloudflared log (last 10 lines) ─────────────────")
if os.path.exists(LOG_CF):
    for line in subprocess.run(f"tail -10 {LOG_CF}", shell=True, capture_output=True, text=True).stdout.splitlines():
        print(f"  {line}")
else:
    print("  (not found — run Cell 4 first)")

print("\n" + "=" * 60)

  Status Check
  Juice Shop : ✅ PID 7
  Port 3000   : ✅ accepting connections
  cloudflared: ✅ PID 5134

  🌐  Public URL  : https://consistency-political-distribute-survival.trycloudflare.com
  📊  Score board : https://consistency-political-distribute-survival.trycloudflare.com/#/score-board

  ── Juice Shop log (last 20 lines) ──────────────────
  info: Configuration production validated (SUCCESS)
  info: Entity models 21 of 21 are initialized (SUCCESS)
  info: All dependencies in ./package.json are satisfied (SUCCESS)
  (node:14326) [DEP0040] DeprecationWarning: The `punycode` module is deprecated. Please use a userland alternative instead.
  (Use `node --trace-deprecation ...` to show where the warning was created)
  info: Required file server.js is present (SUCCESS)
  info: Required file index.html is present (SUCCESS)
  info: Required file styles.css is present (SUCCESS)
  info: Required file main.js is present (SUCCESS)
  info: Required file polyfills.js is present (SUCCESS)
  in

## 📚 Cell 6 — Challenge Quick-Reference (OWASP Top 10 Mapped)

Curated challenges mapped to OWASP Top 10 (2021), with v20's new AI/LLM challenges.  
> **Spoiler warning** — skip if you want to discover challenges organically.

In [20]:
# ============================================================
# Cell 6: Challenge Quick-Reference
# Stars: ⭐ Easy  ⭐⭐ Medium  ⭐⭐⭐ Hard  ⭐⭐⭐⭐ Expert
# ============================================================

CHALLENGES = {
    "A01 — Broken Access Control": [
        ("⭐",    "Access the administration section without logging in",
                  "Browse to /#/administration directly"),
        ("⭐⭐",   "View another user's basket",
                  "Manipulate basket ID in the REST API request"),
        ("⭐⭐",   "Access the forbidden /ftp directory",
                  "Append /ftp to the base URL — directory listing exposed"),
        ("⭐⭐⭐",  "Forge a coupon for 80%+ discount",
                  "Reverse the coupon generation logic in client-side JS"),
    ],
    "A02 — Cryptographic Failures": [
        ("⭐",    "Find the confidential document in /ftp",
                  "Browse to /ftp — directory listing is exposed"),
        ("⭐⭐",   "Retrieve the database backup file",
                  "Null-byte injection (%00.md) in /ftp to bypass extension check"),
        ("⭐⭐⭐",  "Crack the admin's forgotten password hash",
                  "The hash is MD5 — use a rainbow table or hashcat"),
    ],
    "A03 — Injection": [
        ("⭐",    "Log in as admin via SQL injection",
                  "Email: ' OR 1=1--   Password: anything"),
        ("⭐⭐",   "Extract the full user table via SQLi",
                  "UNION SELECT on the product search endpoint"),
        ("⭐⭐⭐",  "NoSQL injection on user search",
                  "Craft JSON body with MongoDB operators ($ne, $gt)"),
        ("⭐⭐⭐",  "OS command injection via file upload",
                  "Embed shell metacharacters in the uploaded filename"),
    ],
    "A05 — Security Misconfiguration": [
        ("⭐",    "Find the hidden easter egg",
                  "Search 'eastere' in the minified frontend JS"),
        ("⭐⭐",   "Access the deprecated B2B interface",
                  "Try /b2b/v2 — it is still live"),
        ("⭐⭐",   "Access Swagger API documentation",
                  "Browse to /api-docs"),
    ],
    "A07 — Identification & Auth Failures": [
        ("⭐",    "Log in with admin default credentials",
                  "admin@juice-sh.op / admin123"),
        ("⭐⭐",   "Reset Jim's password without knowing it",
                  "His security question answer is Star Trek canon"),
        ("⭐⭐",   "Brute-force a 4-digit pin",
                  "No lockout by default — iterate 0000–9999"),
        ("⭐⭐⭐",  "Forge a JWT to escalate to admin",
                  "Change alg to 'none' and strip the signature"),
    ],
    "A08 — Software & Data Integrity": [
        ("⭐⭐",   "Upload a malicious file via the profile photo endpoint",
                  "MIME type check is client-side only — bypass with curl"),
        ("⭐⭐⭐",  "Identify a vulnerable dependency",
                  "Run npm audit or check against Snyk from the challenge UI"),
    ],
    "A09 — Security Logging & Monitoring Failures": [
        ("⭐",    "Trigger a server error not properly logged",
                  "Send malformed input to cause an unhandled 500"),
        ("⭐⭐",   "Confirm lack of brute-force alerting",
                  "Submit 10+ failed logins — no lockout or alert fires"),
    ],
    "A10 — SSRF": [
        ("⭐⭐⭐",  "Exploit SSRF in the coupon redemption flow",
                  "Direct the coupon URL parameter at an internal endpoint"),
    ],
    "✨ NEW in v20 — AI/LLM Challenges (requires LLM endpoint config)": [
        ("⭐⭐",   "Chatbot Prompt Injection",
                  "Override the chatbot system prompt via crafted user input"),
        ("⭐⭐⭐",  "Greedy Chatbot Manipulation",
                  "Socially engineer the chatbot into granting an unauthorized discount"),
        ("⭐⭐",   "AI Debugging",
                  "Exploit chatbot debug mode to exfiltrate system instructions"),
    ],
    "Bonus — XSS & Client-Side": [
        ("⭐",    "Reflected XSS via the search bar",
                  "<iframe src=\"javascript:alert('xss')\">"),
        ("⭐⭐",   "Stored XSS in a product review",
                  "Post a review containing a <script> tag"),
        ("⭐⭐⭐",  "DOM-based XSS via the redirect parameter",
                  "Craft a login URL with a javascript: scheme in the redirect param"),
    ],
}

print("=" * 70)
print("  OWASP Juice Shop v20.0.0 — Challenge Quick-Reference")
print("  Stars: ⭐ Easy  ⭐⭐ Medium  ⭐⭐⭐ Hard")
print("=" * 70)

for category, challenges in CHALLENGES.items():
    print(f"\n  ▸ {category}")
    print(f"  {'─'*66}")
    for stars, name, hint in challenges:
        print(f"  {stars:<4}  {name}")
        print(f"         💡 {hint}")

print("\n" + "=" * 70)
print("  Full companion guide : https://pwning.owasp-juice.shop/companion-guide")
print("  LLM endpoint setup   : https://howto-llm.owasp-juice.shop")
print("  Score board          : <your-url>/#/score-board")
print("=" * 70)

  OWASP Juice Shop v20.0.0 — Challenge Quick-Reference
  Stars: ⭐ Easy  ⭐⭐ Medium  ⭐⭐⭐ Hard

  ▸ A01 — Broken Access Control
  ──────────────────────────────────────────────────────────────────
  ⭐     Access the administration section without logging in
         💡 Browse to /#/administration directly
  ⭐⭐    View another user's basket
         💡 Manipulate basket ID in the REST API request
  ⭐⭐    Access the forbidden /ftp directory
         💡 Append /ftp to the base URL — directory listing exposed
  ⭐⭐⭐   Forge a coupon for 80%+ discount
         💡 Reverse the coupon generation logic in client-side JS

  ▸ A02 — Cryptographic Failures
  ──────────────────────────────────────────────────────────────────
  ⭐     Find the confidential document in /ftp
         💡 Browse to /ftp — directory listing is exposed
  ⭐⭐    Retrieve the database backup file
         💡 Null-byte injection (%00.md) in /ftp to bypass extension check
  ⭐⭐⭐   Crack the admin's forgotten password hash
         💡 The h

## 🛑 Cell 7 — Shutdown

In [31]:
# ============================================================
# Cell 7: Graceful Shutdown
# ============================================================
import subprocess, builtins

print("=" * 60)
print("  Shutting down Juice Shop lab...")

# Terminate via stored process handle if available
for attr, label in [("JS_PROC", "Juice Shop"), ("CF_PROC", "cloudflared")]:
    proc = getattr(builtins, attr, None)
    if proc and proc.poll() is None:
        proc.terminate()
        print(f"  ✅  {label} (PID {proc.pid}) terminated.")

# pkill as belt-and-suspenders fallback
for pat, label in [
    ("npm.*start",           "npm start"),
    ("node.*bin/juice-shop", "juice-shop node"),
    ("cloudflared",          "cloudflared"),
]:
    r = subprocess.run(f"pkill -f '{pat}' 2>/dev/null", shell=True)
    if r.returncode == 0:
        print(f"  ✅  {label} stopped via pkill.")

print()
print("  Lab shutdown complete.")
print("  Re-run Cell 4 to start a new session.")
print("=" * 60)


  Shutting down Juice Shop lab...
  ✅  Juice Shop (PID 9808) terminated.
  ✅  cloudflared (PID 9844) terminated.

  Lab shutdown complete.
  Re-run Cell 4 to start a new session.
